In [8]:
#!/usr/bin/env julia
"""
collect_exercises.jl

Collects all exercises from lecture notebooks (L##.ipynb) and writes them
to a single Quarto .qmd output file, using .exercise-box div syntax
compatible with random-2.ipynb.

Skips L00.ipynb.
Embeds all code cell outputs (plots, text, HTML) as inline content.

Usage:
    julia collect_exercises.jl
"""

using JSON

# ── Configuration ─────────────────────────────────────────────────────────────
lectures_dir = "../lectures"
output_file  = "../exercises/exercises.qmd"

# ── Helpers ───────────────────────────────────────────────────────────────────

"""
    cell_to_markdown(cell, lang) -> String

Convert a single notebook cell to a Markdown string.
- Markdown cells: returned as-is.
- Code cells: source in a collapsible fenced block, followed by all outputs:
  image/png as base64 img, text/html as raw HTML,
  text/plain and stream as fenced code blocks.
"""
function cell_to_markdown(cell, lang::String="julia")::String
    buf = IOBuffer()
    src = join(cell["source"])
    isempty(strip(src)) && return ""

    if cell["cell_type"] == "markdown"
        println(buf, src)
    else  # code / raw
        println(buf, "<details><summary>Code</summary>\n")
        println(buf, "```$lang")
        println(buf, src)
        println(buf, "```")
        println(buf, "</details>\n")

        # Embed all outputs produced by this cell
        for output in get(cell, "outputs", [])
            data = get(output, "data", Dict())

            if haskey(data, "image/png")
                b64 = replace(string(data["image/png"]), '\n' => "")
                println(buf, "\n<img src=\"data:image/png;base64,$b64\" style=\"max-width:100%;height:auto;display:block;margin:.5em 0;\">")

            elseif haskey(data, "text/html")
                html = join(data["text/html"])
                println(buf, "\n$html")

            elseif haskey(data, "text/plain")
                txt = join(data["text/plain"])
                println(buf, "\n```\n$txt\n```")

            elseif get(output, "output_type", "") == "stream"
                txt = join(get(output, "text", []))
                isempty(strip(txt)) || println(buf, "\n```\n$txt\n```")
            end
        end
    end

    return String(take!(buf))
end

"""
    notebook_to_markdown(path) -> String

Convert a Jupyter notebook to a single Markdown string.
"""
function notebook_to_markdown(path::String)::String
    nb   = JSON.parsefile(path)
    lang = get(get(get(nb, "metadata", Dict()), "kernelspec", Dict()), "language", "julia")
    buf  = IOBuffer()
    for cell in nb["cells"]
        md = cell_to_markdown(cell, lang)
        isempty(md) && continue
        println(buf, md)
    end
    return String(take!(buf))
end

"""
    extract_exercises(markdown, lecture_label) -> Vector{String}

Pull every `::: {#exr-...}` block out of markdown and re-wrap as
`::: {.exercise-box #bank-exr-...}` with a bold cross-reference label.
Nested `:::` blocks (hints, solutions) are preserved.
Trailing outputs immediately after closing ::: are pulled into the block.
"""
function extract_exercises(markdown::String, lecture_label::String)::Vector{String}
    exercises = String[]
    lines     = split(markdown, '\n')
    n         = length(lines)
    i         = 1

    while i <= n
        line = lines[i]
        m = match(r"^:::\s*\{#(exr-[\w-]+)", line)
        if m !== nothing
            exr_id = m.captures[1]
            depth  = 1
            i     += 1

            # Collect inner lines until matching closing :::
            inner = String[]
            while i <= n && depth > 0
                l = lines[i]
                if occursin(r"^:::\s*\{", l)
                    depth += 1
                    push!(inner, l)
                elseif strip(l) == ":::"
                    depth -= 1
                    depth > 0 && push!(inner, l)
                else
                    push!(inner, l)
                end
                i += 1
            end

            # Remove bank/practice links (redundant inside the bank)
            inner = filter(l -> !occursin("bank.html", l) && !occursin("random-2.html", l), inner)

            # Strip leading/trailing blank lines
            while !isempty(inner) && isempty(strip(first(inner))); popfirst!(inner); end
            while !isempty(inner) && isempty(strip(last(inner)));  pop!(inner);      end

            # Remove equation numbers
            inner = map(inner) do line
                line = replace(line, r"\s*\{#eq-[\w-]+\}" => "")
                line = replace(line, r"\\tag\{[^}]*\}"    => "")
                line
            end

            # Peek ahead: grab outputs immediately following closing :::
            trailing = String[]
            j = i
            while j <= n
                l = strip(lines[j])
                if isempty(l)
                    j += 1
                elseif startswith(l, "<img ") || (startswith(l, "<") && !startswith(l, "<details>") && !startswith(l, "</details>"))
                    push!(trailing, lines[j])
                    j += 1
                elseif l == "```"
                    push!(trailing, lines[j]); j += 1
                    while j <= n && strip(lines[j]) != "```"
                        push!(trailing, lines[j]); j += 1
                    end
                    j <= n && (push!(trailing, lines[j]); j += 1)
                else
                    break
                end
            end
            if !isempty(trailing)
                i = j
                append!(inner, ["", trailing...])
            end

            block = "<!-- $lecture_label -->\n" *
                    "::: {.exercise-box #bank-$exr_id}\n" *
                    "**@$exr_id**\n\n" *
                    join(inner, '\n') * "\n" *
                    ":::"

            push!(exercises, block)
        else
            i += 1
        end
    end
    return exercises
end

# ── Main ──────────────────────────────────────────────────────────────────────

function main()
    pattern   = r"^L\d+\.ipynb$"
    all_files = filter(f -> occursin(pattern, f), readdir(lectures_dir))
    sort!(all_files)
    all_files = filter(f -> f != "L00.ipynb", all_files)

    if isempty(all_files)
        @warn "No lecture notebooks found in $lectures_dir"
        return
    end

    println("Found $(length(all_files)) lecture notebook(s): $(join(all_files, ", "))")

    mkpath(dirname(output_file))
    open(output_file, "w") do io
        write(io, """
---
title: "Exercises"
format:
  html:
    code-copy: true
    toc: true
---

""")

        for fname in all_files
            label   = splitext(fname)[1]
            num_str = replace(label, r"^L0*" => "")
            lec_num = tryparse(Int, num_str)

            path      = joinpath(lectures_dir, fname)
            markdown  = notebook_to_markdown(path)
            exercises = extract_exercises(markdown, label)

            if isempty(exercises)
                println("  $label: no exercises found")
                continue
            end

            println("  $label: $(length(exercises)) exercise(s)")
            println(io, "## Lecture $lec_num\n")
            for ex in exercises
                println(io, ex)
                println(io)
            end
        end
    end

    println("\nWrote exercises to: $output_file")
end

main()


Found 4 lecture notebook(s): L01.ipynb, L02.ipynb, L03.ipynb, L04.ipynb
  L01: 6 exercise(s)
  L02: 9 exercise(s)
  L03: 10 exercise(s)
  L04: 6 exercise(s)

Wrote exercises to: ../exercises/exercises.qmd
